# 03 Merge: OpenCity results + MyNeta affidavits + TCPD party table (rewritten)

**Inputs**
- `opencity_results_clean_2026-10-20.csv`: general election results (notebook 02)
- `opencity_byelections_2014_onwards.csv`: by-election results (notebook 01)
- `candidates_processed.csv`: MyNeta candidates, general and by-election rows (scraper + OCR)
- `TCPD-PPI_1962_2021.csv`: TCPD party table

**Outputs** (in `OUT_DIR`): merged general table, merged by-election table, combined table, MyNeta rows without a partner, party map.

**What changed from the old version**
1. Every text column of every input is cleaned first: non-breaking spaces, newlines, tabs and repeated spaces are collapsed, ends trimmed. The number of changed cells is printed per column.
2. Asset and liability strings are parsed with a new parser. The old one stopped at the first space, so OCR strings like `Rs 8,01 ,25,802 ~ 8 Crores` became **801** instead of 80,125,802. The new parser drops all spaces inside the number and checks it against the `~ 8 Crore` approximation that MyNeta prints next to it.
3. Impossible ages (0) become empty and are flagged.
4. Files are found in this folder or in `raw/`, `interim/`, `clean/` (one level up or here), so the notebook runs in either layout.
5. A final checks section asserts row counts, one-to-one matching, one winner per seat, numeric types, clean text and explained nulls.

The candidate matching itself (seat lookup, TCPD party mapping, six name rules) is kept from the old version, because its results checked out (winner flag agreed on every matched pair).

In [1]:
import os
import re
import difflib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

### Step 0: find the input files

In [2]:
COLLECTION_DATE = "2026-10-20"
STATES = ["Bihar", "Maharashtra", "Uttar Pradesh", "Tamil Nadu"]

NB_DIR = Path.cwd()
DATA_DIR = Path(os.environ.get("DATA_DIR", NB_DIR))          # set DATA_DIR to point somewhere else
SEARCH_DIRS = [DATA_DIR] + [base / sub for base in (DATA_DIR, DATA_DIR.parent) for sub in ("raw", "interim", "clean")]

def find_file(name):
    for folder in SEARCH_DIRS:
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(f"{name} not found in: {[str(f) for f in SEARCH_DIRS]}")

default_out = DATA_DIR.parent / "clean" if (DATA_DIR.parent / "clean").is_dir() else DATA_DIR / "clean"
OUT_DIR = Path(os.environ.get("OUT_DIR", default_out))
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "results": find_file(f"opencity_results_clean_{COLLECTION_DATE}.csv"),
    "oc_bye": find_file("opencity_byelections_2014_onwards.csv"),
    "myneta": find_file("candidates_processed.csv"),
    "ppi": find_file("TCPD-PPI_1962_2021.csv"),
}
for k, p in paths.items():
    print(f"{k:8} {p}")
print("outputs ->", OUT_DIR)

results  /Users/pravas/Desktop/DA25B051/DCV-Project/clean/opencity_results_clean_2026-10-20.csv
oc_bye   /Users/pravas/Desktop/DA25B051/DCV-Project/interim/opencity_byelections_2014_onwards.csv
myneta   /Users/pravas/Desktop/DA25B051/DCV-Project/interim/candidates_processed.csv
ppi      /Users/pravas/Desktop/DA25B051/DCV-Project/interim/TCPD-PPI_1962_2021.csv
outputs -> /Users/pravas/Desktop/DA25B051/DCV-Project/clean


In [3]:
results = pd.read_csv(paths["results"])
oc_bye = pd.read_csv(paths["oc_bye"], low_memory=False)
myneta = pd.read_csv(paths["myneta"])
ppi = pd.read_csv(paths["ppi"])

# row numbers in the input files (0 = first data row under the header), carried to the output
results["OC_Row"] = results.index
oc_bye["OC_Row"] = oc_bye.index
myneta["MN_Row"] = myneta.index

N_RESULTS, N_MYNETA = len(results), len(myneta)
print("OpenCity general:", results.shape, "| OpenCity by-elections (all states):", oc_bye.shape)
print("MyNeta:", myneta.shape, "| TCPD party table:", ppi.shape)

OpenCity general: (10378, 24) | OpenCity by-elections (all states): (410, 46)
MyNeta: (10395, 16) | TCPD party table: (10153, 21)


### Step 1: clean the text in every input

MyNeta asset strings carry non-breaking spaces (`\xa0`) and line breaks from OCR (`"Rs 1,60,59,806\n~1 Crore+."`). One function is used on every text column of every file: special spaces and line breaks become normal spaces, runs of spaces become one, ends are trimmed, empty strings become missing.

In [4]:
SPACE_CHARS = re.compile(r"[\s  -​  　﻿]+")

def clean_text(value):
    if pd.isna(value):
        return value
    text = SPACE_CHARS.sub(" ", str(value)).strip()
    return text if text else np.nan

def text_columns(frame):
    return [c for c in frame.columns if pd.api.types.is_string_dtype(frame[c]) or frame[c].dtype == object]

def clean_frame(frame, label):
    report = {}
    for col in text_columns(frame):
        before = frame[col]
        after = before.map(clean_text)
        changed = int((before.fillna("<NA>").astype(str) != after.fillna("<NA>").astype(str)).sum())
        if changed:
            report[col] = changed
        frame[col] = after
    print(f"{label}: cells changed per column -> {report or 'none'}")
    return frame

myneta = clean_frame(myneta, "MyNeta")
results = clean_frame(results, "OpenCity general")
oc_bye = clean_frame(oc_bye, "OpenCity by-elections")
ppi = clean_frame(ppi, "TCPD party table")

MyNeta: cells changed per column -> {'total_assets': 10308, 'liabilities': 8578}
OpenCity general: cells changed per column -> none
OpenCity by-elections: cells changed per column -> none
TCPD party table: cells changed per column -> {'Party_Name': 324, 'Abbreviations': 1}


### Step 2: duplicates and impossible values in the inputs

In [5]:
print("exact duplicate rows | MyNeta:", int(myneta.duplicated().sum()),
      "| OpenCity general:", int(results.duplicated().sum()), "| by-elections:", int(oc_bye.duplicated().sum()))

mn_key = ["year", "state", "const", "sno", "is_byelection", "bye_election_date"]
oc_key = ["Year", "State", "Constituency_No", "Person_ID"]
print("duplicate MyNeta keys", mn_key, ":", int(myneta.duplicated(mn_key).sum()))
print("duplicate OpenCity keys", oc_key, ":", int(results.duplicated(oc_key).sum()))

# same name twice in one seat is allowed (namesakes) but shown
namesakes = myneta[myneta.duplicated(["year", "state", "const", "candidate", "is_byelection"], keep=False)]
print("MyNeta namesake rows (same name, same seat, different person):", len(namesakes))

# ages: Lok Sabha minimum is 25; 0 means the scraper got no age
myneta["Age_Missing"] = myneta["age"].le(0)
myneta.loc[myneta["Age_Missing"], "age"] = np.nan
print("ages set to empty (were 0):", int(myneta["Age_Missing"].sum()),
      "| ages under 25 kept as listed:", myneta.loc[myneta["age"] < 25, "age"].tolist())
print("criminal cases min/max:", myneta["criminal_cases"].min(), myneta["criminal_cases"].max())

exact duplicate rows | MyNeta: 0 | OpenCity general: 0 | by-elections: 0
duplicate MyNeta keys ['year', 'state', 'const', 'sno', 'is_byelection', 'bye_election_date'] : 0
duplicate OpenCity keys ['Year', 'State', 'Constituency_No', 'Person_ID'] : 0
MyNeta namesake rows (same name, same seat, different person): 26
ages set to empty (were 0): 39 | ages under 25 kept as listed: [24.0, 21.0]
criminal cases min/max: 0 382


### Step 3: candidate names

Same as before: upper case, symbols become spaces, titles (`Dr`, `Adv`, `Thiru`, `Smt`...) dropped. Three versions are kept for matching:
- `Name`: cleaned name
- `Name_Key`: words in alphabetical order (ignores word order)
- `Letters_Key`: all letters sorted, spaces ignored (`K P MUNUSAMY` equals `KP MUNUSAMY`)

In [6]:
TITLES = {"DR", "PROF", "PROFESSOR", "ADV", "ADVOCATE", "ER", "ENGR", "SHRI", "SRI", "SHREE", "SMT", "SMTI", "SHRIMATI",
          "TMT", "THIRU", "THIRUMATHI", "THIRUVALAR", "SELVI", "MR", "MRS", "MS", "MISS", "COL", "CAPT", "MAJ", "RETD", "RTD"}

def clean_name(name):
    text = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper())
    return " ".join(t for t in text.split() if t not in TITLES)

def add_name_keys(frame, col):
    frame["Name"] = frame[col].map(clean_name)
    frame["Name_Key"] = frame["Name"].map(lambda n: " ".join(sorted(n.split())))
    frame["Letters_Key"] = frame["Name"].map(lambda n: "".join(sorted(n.replace(" ", ""))))
    return frame

myneta = add_name_keys(myneta, "candidate")
results = add_name_keys(results, "Candidate")
print("empty names after cleaning | MyNeta:", int((myneta["Name"] == "").sum()), "| OpenCity:", int((results["Name"] == "").sum()))

empty names after cleaning | MyNeta: 0 | OpenCity: 0


### Step 4: link MyNeta seats to constituency numbers

MyNeta gives the seat name (`GAYA (SC)`); OpenCity has the number. Names are compared without `(SC)`/`(ST)` and symbols, inside the state; leftovers are matched by similar spelling and printed.

In [7]:
def seat_norm(s):
    s = re.sub(r"\((SC|ST|GEN)\)", "", str(s), flags=re.I)
    return re.sub(r"[^A-Z0-9]", "", s.upper())

seats = results.drop_duplicates(["State", "Constituency_No"])[["State", "Constituency_No", "Constituency_Name", "Constituency_Type"]]
seat_lookup = {(r.State, seat_norm(r.Constituency_Name)): r.Constituency_No for r in seats.itertuples()}
assert len(seat_lookup) == len(seats), "two seats share a normalised name"

myneta["State"] = myneta["state"].str.title()
print("MyNeta states:", sorted(myneta["State"].unique()))
assert set(myneta["State"]) <= set(STATES)

seat_pairs = myneta[["State", "const"]].drop_duplicates()
resolved, fuzzy_seats, unresolved = {}, [], []
for state, const in seat_pairs.itertuples(index=False):
    key = (state, seat_norm(const))
    if key in seat_lookup:
        resolved[(state, const)] = seat_lookup[key]
        continue
    same_state = [k[1] for k in seat_lookup if k[0] == state]
    close = difflib.get_close_matches(key[1], same_state, n=1, cutoff=0.8)
    if close:
        resolved[(state, const)] = seat_lookup[(state, close[0])]
        fuzzy_seats.append((state, const, close[0]))
    else:
        unresolved.append((state, const))

myneta["Constituency_No"] = [resolved.get((s, c)) for s, c in zip(myneta["State"], myneta["const"])]
myneta["Constituency_No"] = myneta["Constituency_No"].astype("Int64")
print("MyNeta seat names:", len(seat_pairs), "| exact:", len(seat_pairs) - len(fuzzy_seats) - len(unresolved))
print("matched by similar spelling:", fuzzy_seats)
print("not matched:", unresolved)
assert not unresolved

MyNeta states: ['Bihar', 'Maharashtra', 'Tamil Nadu', 'Uttar Pradesh']
MyNeta seat names: 255 | exact: 250
matched by similar spelling: [('Maharashtra', 'AHMEDNAGAR', 'AHMADNAGAR'), ('Maharashtra', 'HATKANANGALE', 'HATKANANGLE'), ('Tamil Nadu', 'TIRUVALLUR (SC)', 'THIRUVALLUR'), ('Tamil Nadu', 'TIRUVALLUR', 'THIRUVALLUR'), ('Uttar Pradesh', 'GAUTAM BUDHHA NAGAR', 'GAUTAMBUDDHANAGAR')]
not matched: []


In [8]:
general_mn_rows = myneta[~myneta["is_byelection"]]
for year in [2014, 2019, 2024]:
    have = set(general_mn_rows.loc[general_mn_rows["year"] == year, ["State", "Constituency_No"]].dropna().itertuples(index=False, name=None))
    missing = [(s, n, name) for s, n, name in seats[["State", "Constituency_No", "Constituency_Name"]].itertuples(index=False) if (s, n) not in have]
    print(year, "seats without MyNeta rows:", len(missing), missing)

2014 seats without MyNeta rows: 0 []
2019 seats without MyNeta rows: 0 []
2024 seats without MyNeta rows: 0 []


### Step 5: party names with the TCPD party table

Unchanged logic. Each party string is turned into a key (upper case, symbols removed) and looked up among TCPD Lok Sabha party names and abbreviations, state first.

| Method | Meaning |
|---|---|
| exact | one live TCPD party fits |
| exact-old | only parties that stopped contesting before 2014 fit |
| alias | hand-written rule (`AIADMK` is `ADMK`; 2022/2023 Shiv Sena, NCP, LJP splits go to the parent) |
| ambiguous | more than one live party fits; the largest is taken |
| fuzzy | 95% similar spelling |
| unmapped | not in TCPD (e.g. parties formed after 2021) |

In [9]:
def pnorm(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())

ppi_ls = ppi[ppi["Assembly"] == "Lok_Sabha"].copy()
ppi_ls["State"] = ppi_ls["State_Name"].str.replace("_", " ")

party_info = ppi_ls.groupby("Party_ID").agg(Party_Name=("Party_Name", "first"), Party_Type=("Party_Type", "first"),
                                            Last_Year=("Last_Year", "max"), Size=("Candidates_Contested", "max"))

name_lookup, abbr_lookup = {}, {}
for r in ppi_ls.itertuples():
    name_lookup.setdefault(pnorm(r.Party_Name), set()).add((r.Party_ID, r.State))
    for text in [r.Frequent_Abbreviation, r.Last_Abbreviation] + str(r.Abbreviations).split("|"):
        key = pnorm(text)
        if key and key != "NAN":
            abbr_lookup.setdefault(key, set()).add((r.Party_ID, r.State))
lookup = {**abbr_lookup, **name_lookup}   # a full name beats an abbreviation

ALIASES = {"AIADMK": "ADMK", "AAP": "AAAP"}
FULL_NAMES = {
    "Bharatiya Janata Party": "BJP", "Indian National Congress": "INC", "Bahujan Samaj Party": "BSP",
    "Samajwadi Party": "SP", "Rashtriya Janata Dal": "RJD", "Janata Dal (United)": "JD(U)",
    "Dravida Munnetra Kazhagam": "DMK", "All India Anna Dravida Munnetra Kazhagam": "ADMK",
    "Nationalist Congress Party": "NCP", "Communist Party of India": "CPI", "Communist Party of India (Marxist)": "CPM",
    "Communist Party of India (Marxist-Leninist) (Liberation)": "CPI(ML)(L)", "Shiv Sena": "SHS",
    "All India Majlis-E-Ittehadul Muslimeen": "AIMIM", "Pattali Makkal Katchi": "PMK",
    "Desiya Murpokku Dravida Kazhagam": "DMDK", "Viduthalai Chiruthaigal Katchi": "VCK",
    "Marumalarchi Dravida Munnetra Kazhagam": "MDMK", "Rashtriya Lok Dal": "RLD", "Naam Tamilar Katchi": "NTK",
    "Indian Union Muslim League": "IUML", "Apna Dal (Soneylal)": "ADAL", "Hindustani Awam Morcha (Secular)": "HAMS",
    "Lok Janshakti Party": "LJP", "Independent": "IND", "None of the Above": "NOTA",
}
ALIASES.update({pnorm(name): pnorm(c) for name, c in FULL_NAMES.items()})
FACTION_RULES = [("UDDHAV", "SHS"), ("SHARADCHANDRA", "NCP"), ("NCPSP", "NCP"), ("RAMVILAS", "LJP"), ("LJPRV", "LJP")]

def resolve_party(raw, state):
    key, method = pnorm(raw), "exact"
    for fragment, parent in FACTION_RULES:
        if fragment in key:
            key, method = parent, "alias"
    if key in ALIASES:
        key, method = ALIASES[key], "alias"
    options = lookup.get(key)
    if not options:
        close = difflib.get_close_matches(key, list(lookup), n=1, cutoff=0.95)
        if not close:
            return np.nan, "unmapped", []
        options, method = lookup[close[0]], "fuzzy"
    ids = {pid for pid, st in options if st in (state, "All States")} or {pid for pid, _ in options}
    live = [p for p in ids if party_info.loc[p, "Last_Year"] >= 2014]
    if len(live) == 1:
        return live[0], method, []
    if len(live) > 1:
        ranked = sorted(live, key=lambda p: party_info.loc[p, "Size"], reverse=True)
        return ranked[0], "ambiguous", ranked
    best = max(ids, key=lambda p: party_info.loc[p, "Size"])
    return best, ("exact-old" if method == "exact" else method), []

In [10]:
results["Party_Raw"] = results["Party_Full"]
myneta["Party_Raw"] = myneta["party"]
oc_bye = oc_bye.rename(columns={"Party_ID": "Party_ID_in_file"})
oc_bye["State"] = oc_bye["State_Name"].str.replace("_", " ")
oc_bye["Party_Raw"] = oc_bye["Party"]

pairs = pd.concat([results[["State", "Party_Raw"]], myneta[["State", "Party_Raw"]], oc_bye[["State", "Party_Raw"]]]).drop_duplicates()
rows = []
for state, raw in pairs.itertuples(index=False):
    pid, method, options = resolve_party(raw, state)
    rows.append({"State": state, "Party_Raw": raw, "Party_ID": pid, "Party_Method": method, "Options": options})
party_map = pd.DataFrame(rows)
party_map["Party_Canonical"] = party_map["Party_ID"].map(party_info["Party_Name"])
party_map["Party_Type"] = party_map["Party_ID"].map(party_info["Party_Type"]).fillna("Not in TCPD")
assert not party_map.duplicated(["State", "Party_Raw"]).any()

print("distinct (state, party string) pairs:", len(party_map))
print(party_map["Party_Method"].value_counts().to_string())

rows_per_string = pd.concat([results["Party_Raw"], myneta["Party_Raw"], oc_bye["Party_Raw"]]).value_counts()
for method in ["ambiguous", "fuzzy", "exact-old"]:
    part = party_map[party_map["Party_Method"] == method].drop_duplicates("Party_Raw")
    print(f"\n{method}: {len(part)} strings (first 10)")
    for r in part.head(10).itertuples():
        print(f"  {r.Party_Raw!r:50} -> {r.Party_Canonical!r} ({r.Party_Type})")
unmapped = party_map[party_map["Party_Method"] == "unmapped"].drop_duplicates("Party_Raw")
print(f"\nunmapped: {len(unmapped)} strings covering {int(rows_per_string.reindex(unmapped['Party_Raw']).sum())} rows; largest:")
print(rows_per_string.reindex(unmapped["Party_Raw"]).sort_values(ascending=False).head(10).to_string())

distinct (state, party string) pairs: 1742
Party_Method
exact        1333
unmapped      277
alias          54
ambiguous      38
fuzzy          32
exact-old       8

ambiguous: 19 strings (first 10)
  'JD(U)'                                            -> 'Janata Dal (United)' (State-based Party)
  'CPI(ML)(L)'                                       -> 'Communist Party of India (Marxist-Leninist) (Liberation)' (Local Party)
  'VIP'                                              -> 'Vanchitsamaj Insaaf Party' (Local Party)
  'CPIM'                                             -> 'COMMUNIST MARXIST PARTY OF INDIA' (State-based Party)
  'ASP'                                              -> 'Ambedkar Samaj Party' (Local Party)
  'RP(K)'                                            -> 'Republican Party of India (Khobragade Group)' (Local Party)
  'RSPS'                                             -> 'Rashtriya Samaj Paksha' (Local Party)
  'MMKA'                                             -> 'Makk

In [11]:
def add_party(frame):
    merged = frame.merge(party_map.drop(columns="Options"), on=["State", "Party_Raw"], how="left", validate="many_to_one")
    assert len(merged) == len(frame)
    merged.index = frame.index
    return merged

results = add_party(results)
myneta = add_party(myneta)
oc_bye = add_party(oc_bye)
print(results.groupby("Year")["Party_Type"].value_counts().unstack(fill_value=0).to_string())

Party_Type  Independents  Local Party  National Party  Not in TCPD  State-based Party
Year                                                                                 
2014                1503          893             307            0                934
2019                1474          426             244          731                447
2024                1698          230             208          799                484


### Step 6: asset and liability strings to rupees

MyNeta prints `Rs 1,60,59,806 ~ 1 Crore+`. OCR adds spaces inside the number (`Rs 8,01 ,25,802`, `Rs 22,85,98, 164`). The parser:
1. takes the text between `Rs` and `~`,
2. keeps only the digits (so spaces and commas inside the number no longer matter),
3. reads the approximation after `~` (`8 Crores`, `16 Lacs+`, `50 Thou+`, `5 Hund+`) and checks that the parsed value has the same leading figure in that unit.

`Nil` becomes 0. Each value gets a status: `ok` (checked against the approximation), `nil`, `no_approx` (no approximation printed, e.g. `Rs 0 ~`), `mismatch` (approximation disagrees), `unreadable`.

In [12]:
UNITS = [("crore", 1e7), ("lac", 1e5), ("lakh", 1e5), ("thou", 1e3), ("hund", 1e2)]

def parse_money(text):
    """returns (rupees, status)"""
    if pd.isna(text):
        return np.nan, "missing"
    t = clean_text(text)
    if t.lower() in {"nil", "none", "0"}:
        return 0.0, "nil"
    m = re.match(r"^\s*Rs\.?\s*([\d,\s]+?)\s*(?:~\s*(.*))?$", t, flags=re.I)
    if not m:
        return np.nan, "unreadable"
    digits = re.sub(r"\D", "", m.group(1))
    if not digits:
        return np.nan, "unreadable"
    value = float(digits)
    approx = (m.group(2) or "").strip()
    a = re.match(r"^([\d.]+)\s*([A-Za-z]+)", approx)
    if not a:
        return value, "no_approx"
    lead, word = float(a.group(1)), a.group(2).lower()
    unit = next((u for k, u in UNITS if word.startswith(k)), None)
    if unit is None:
        return value, "mismatch"
    return value, ("ok" if int(value // unit) == int(lead) else "mismatch")

def parse_rs_old(text):
    # the parser of the old notebook, kept only to show what changed
    t = str(text).replace("\xa0", " ").strip()
    if t.lower() == "nil":
        return 0.0
    found = re.search(r"Rs\s*([\d,]+)", t)
    return float(found.group(1).replace(",", "")) if found else np.nan

# tests on strings seen in the data
for s, expected in [("Rs 1,60,59,806 ~1 Crore+.", 16059806), ("Rs 8,01 ,25,802 ~ 8 Crores.", 80125802),
                    ("Rs 22,85,98, 164 ~ 22 Crores.", 228598164), ("Rs 0 ~", 0), ("Rs 0", 0), ("Nil", 0),
                    ("Rs 5,000 ~ 5 Thou+", 5000), ("Rs 4,91 ,603 ~4Lacs+.", 491603), ("Rs 500 ~ 5 Hund+", 500)]:
    got = parse_money(s)[0]
    assert got == expected, (s, got)
print("parser tests passed")

parser tests passed


In [13]:
raw_assets = pd.read_csv(paths["myneta"], usecols=["total_assets", "liabilities"])   # untouched strings, for the comparison
for col, new in [("total_assets", "Assets_Rs"), ("liabilities", "Liabilities_Rs")]:
    parsed = myneta[col].map(parse_money)
    myneta[new] = [v for v, _ in parsed]
    myneta[new + "_Status"] = [s for _, s in parsed]
    print(f"\n{new}: status counts", myneta[new + "_Status"].value_counts().to_dict())

    old = raw_assets[col].map(parse_rs_old)
    fixed = myneta[new].ne(old) & myneta[new].notna()
    print(f"{new}: values the old notebook read wrongly: {int(fixed.sum())}")
    show = pd.DataFrame({"string": raw_assets.loc[fixed, col].map(repr), "old": old[fixed], "new": myneta.loc[fixed, new]})
    print(show.head(20).to_string())

myneta["Assets_Nil"] = myneta["Assets_Rs_Status"].eq("nil")
for col in ["Assets_Rs", "Liabilities_Rs"]:
    bad = myneta[myneta[col + "_Status"].isin(["mismatch", "unreadable", "missing"])]
    print(f"{col}: rows to look at -> {len(bad)}")
    if len(bad):
        print(bad[["year", "const", "candidate", col.replace('_Rs', '').lower().replace('assets', 'total_assets'), col]].to_string())

print("\nassets with no approximation (very small values, kept as listed):")
print(myneta.loc[myneta["Assets_Rs_Status"].eq("no_approx"), ["year", "const", "candidate", "total_assets", "Assets_Rs"]].to_string())
print("\n", myneta[["Assets_Rs", "Liabilities_Rs"]].describe().round(0).to_string())


Assets_Rs: status counts {'ok': 10304, 'nil': 86, 'no_approx': 5}
Assets_Rs: values the old notebook read wrongly: 16
                                 string       old          new
705      'Rs 8,01 ,25,802\n~ 8 Crores.'     801.0   80125802.0
792    'Rs 22,85,98, 164\n~ 22 Crores.'  228598.0  228598164.0
2569     'Rs 5,01 ,84,324\n~ 5 Crores.'     501.0   50184324.0
2879   'Rs 83,82,09, 103\n~ 83 Crores.'  838209.0  838209103.0
2905     'Rs 4,51 ,86,888\n~ 4 Crore+.'     451.0   45186888.0
3066   'Rs 35,59,66, 144\n~ 35 Crores.'  355966.0  355966144.0
5386     'Rs 2,91 ,62,302\n~ 2 Crores.'     291.0   29162302.0
5507           'Rs 4,91 ,603\n~4Lacs+.'     491.0     491603.0
5905    'Rs 12,63,56, 142\n~12 Crores.'  126356.0  126356142.0
6256   'Rs 37,78,59, 166\n~ 37 Crores.'  377859.0  377859166.0
6280     'Rs 4,81 ,45,467\n~ 4 Crores.'     481.0   48145467.0
6377     'Rs 2,61 ,46,972\n~ 2 Crores.'     261.0   26146972.0
6556     'Rs 4,61 ,46,135\n~ 4 Crores.'     461.0   46146135.0

### Step 7: match candidates inside each seat

Unchanged rules. For each (election, seat), OpenCity and MyNeta candidates are paired one-to-one (a row is used once):

1. same name key (same party first when two share a name)
2. same letters key
3. initials + surname on one side, full name on the other
4. name similarity at least 0.85
5. same party (not independent) and similarity at least 0.70
6. the last one left on both sides, same party (not independent)
7. **new:** the seat winner on both sides, if neither is paired yet and the parties do not clash. Both files agree on who won, so this is safe; it catches names written very differently (`A T NANA PATIL` vs `Patil Ashok Tapiram`, `HEMA MALINI` vs `Deol Hema Malini Dharmendra`).

Rules 1 to 5 are not applied when both TCPD parties are known, different, and one of them is a national or state party.

In [14]:
IND_ID = party_info.index[party_info["Party_Type"] == "Independents"][0]
BIG_TYPES = {"National Party", "State-based Party"}

def similarity(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def same_party(o, m):
    return pd.notna(o) and pd.notna(m) and o == m and o != IND_ID

def party_conflict(o, m):
    if pd.isna(o) or pd.isna(m) or o == m:
        return False
    return bool({party_info.at[o, "Party_Type"], party_info.at[m, "Party_Type"]} & BIG_TYPES)

def initials_match(short, long):
    ts, tl = short.split(), long.split()
    full = [t for t in ts if len(t) > 1]
    initials = [t for t in ts if len(t) == 1]
    if not full or not initials or len(tl) < len(ts):
        return False
    rest = list(tl)
    for t in full:
        best = max(rest, key=lambda w: similarity(t, w))
        if similarity(t, best) < 0.85:
            return False
        rest.remove(best)
    firsts = [w[0] for w in rest]
    for i in initials:
        if i not in firsts:
            return False
        firsts.remove(i)
    return True

def match_seat(oc, mn):
    pairs, used_oc, used_mn = [], set(), set()
    def take(o, m, rule, score):
        pairs.append((o, m, rule, score)); used_oc.add(o); used_mn.add(m)

    for rule, col in [(1, "Name_Key"), (2, "Letters_Key")]:
        for m in mn.index:
            if m in used_mn:
                continue
            found = [o for o in oc.index if o not in used_oc and oc.at[o, col] == mn.at[m, col]
                     and not party_conflict(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])]
            if found:
                preferred = [o for o in found if pd.notna(oc.at[o, "Party_ID"]) and oc.at[o, "Party_ID"] == mn.at[m, "Party_ID"]]
                take((preferred or found)[0], m, rule, 1.0)

    scored = []
    for m in mn.index:
        for o in oc.index:
            if m in used_mn or o in used_oc:
                continue
            if party_conflict(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"]):
                continue
            ratio = similarity(oc.at[o, "Name_Key"], mn.at[m, "Name_Key"])
            party_ok = same_party(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])
            a, b = oc.at[o, "Name"], mn.at[m, "Name"]
            if initials_match(a, b) or initials_match(b, a):
                scored.append((0.95 + 0.05 * party_ok, o, m, 3, max(ratio, 0.95)))
            elif ratio >= 0.85:
                scored.append((ratio + 0.05 * party_ok, o, m, 4, ratio))
            elif party_ok and ratio >= 0.70:
                scored.append((ratio, o, m, 5, ratio))
    for _, o, m, rule, ratio in sorted(scored, reverse=True):
        if o not in used_oc and m not in used_mn:
            take(o, m, rule, ratio)

    left_oc = [o for o in oc.index if o not in used_oc]
    left_mn = [m for m in mn.index if m not in used_mn]
    if len(left_oc) == 1 and len(left_mn) == 1 and same_party(oc.at[left_oc[0], "Party_ID"], mn.at[left_mn[0], "Party_ID"]):
        take(left_oc[0], left_mn[0], 6, similarity(oc.at[left_oc[0], "Name_Key"], mn.at[left_mn[0], "Name_Key"]))

    # rule 7: both sources name the seat winner, neither has been paired, and the parties do not clash
    win_oc = [o for o in oc.index if o not in used_oc and oc.at[o, "Winner"]]
    win_mn = [m for m in mn.index if m not in used_mn and mn.at[m, "is_winner"]]
    if len(win_oc) == 1 and len(win_mn) == 1 and not party_conflict(oc.at[win_oc[0], "Party_ID"], mn.at[win_mn[0], "Party_ID"]):
        take(win_oc[0], win_mn[0], 7, similarity(oc.at[win_oc[0], "Name_Key"], mn.at[win_mn[0], "Name_Key"]))
    return pairs

def run_matching(oc_frame, mn_frame, keys):
    oc_groups = {k: g for k, g in oc_frame.groupby(keys)}
    all_pairs, mn_unmatched, oc_unmatched = [], [], []
    seen = set()
    for k, mn_g in mn_frame.groupby(keys):
        seen.add(k)
        if k not in oc_groups:
            mn_unmatched.extend(mn_g.index)
            continue
        oc_g = oc_groups[k]
        p = match_seat(oc_g, mn_g)
        all_pairs.extend(p)
        mn_unmatched.extend(set(mn_g.index) - {x[1] for x in p})
        oc_unmatched.extend(set(oc_g.index) - {x[0] for x in p})
    for k, oc_g in oc_groups.items():          # seats with OpenCity rows but no MyNeta rows at all
        if k not in seen:
            oc_unmatched.extend(oc_g.index)
    return all_pairs, sorted(mn_unmatched), sorted(oc_unmatched)

In [15]:
mn_general = myneta[~myneta["is_byelection"]].rename(columns={"year": "Year"}).copy()
mn_general["Constituency_No"] = mn_general["Constituency_No"].astype(int)
keys = ["Year", "State", "Constituency_No"]

g_pairs, g_mn_left, g_oc_left = run_matching(results, mn_general, keys)
print("OpenCity rows:", len(results), "| MyNeta general rows:", len(mn_general))
print("matched pairs:", len(g_pairs), "| MyNeta rows left:", len(g_mn_left), "| OpenCity rows left:", len(g_oc_left))
print("pairs by rule:", pd.Series([p[2] for p in g_pairs]).value_counts().sort_index().to_dict())

check = pd.DataFrame(g_pairs, columns=["oc", "mn", "rule", "score"])
check["oc_winner"] = results.loc[check["oc"], "Winner"].values
check["mn_winner"] = mn_general.loc[check["mn"], "is_winner"].values
check["oc_party"] = results.loc[check["oc"], "Party_ID"].values
check["mn_party"] = mn_general.loc[check["mn"], "Party_ID"].values
both = check[check["oc_party"].notna() & check["mn_party"].notna()]
print("winner flag agrees:", round((check["oc_winner"] == check["mn_winner"]).mean(), 4))
print("TCPD party agrees:", round((both["oc_party"] == both["mn_party"]).mean(), 4), "of", len(both), "pairs with both known")
print("\npairs and lowest score by rule:")
print(check.groupby("rule").agg(pairs=("oc", "size"), min_score=("score", "min"),
                                 party_agree=("oc_party", lambda s: 0)).drop(columns="party_agree").round(2).to_string())

OpenCity rows: 10378 | MyNeta general rows: 10228
matched pairs: 9966 | MyNeta rows left: 262 | OpenCity rows left: 412
pairs by rule: {1: 8963, 2: 151, 3: 82, 4: 517, 5: 195, 6: 49, 7: 9}
winner flag agrees: 1.0
TCPD party agrees: 0.9975 of 9426 pairs with both known

pairs and lowest score by rule:
      pairs  min_score
rule                  
1      8963       1.00
2       151       1.00
3        82       0.95
4       517       0.85
5       195       0.70
6        49       0.33
7         9       0.36


In [16]:
def show_leftovers(oc_frame, mn_frame, mn_left, oc_left, keys, year_col):
    oc_groups = {k: g.loc[g.index.isin(oc_left)] for k, g in oc_frame.groupby(keys)}
    rows = []
    for m in mn_left:
        row = mn_frame.loc[m]
        k = tuple(row[c] for c in keys)
        cands = oc_groups.get(k)
        best, ratio = None, 0.0
        if cands is not None and len(cands):
            ratio, best = max((similarity(row["Name_Key"], c), i) for i, c in cands["Name_Key"].items())
        rows.append({"Year": row[year_col], "Seat": row["const"], "MyNeta name": row["Name"], "MyNeta party": row["Party_Raw"],
                     "MyNeta winner": row["is_winner"],
                     "Best OpenCity name": oc_frame.at[best, "Name"] if best is not None else None,
                     "OC party": oc_frame.at[best, "Party_Raw"] if best is not None else None,
                     "similarity": round(ratio, 2)})
    table = pd.DataFrame(rows)
    return table.sort_values("similarity", ascending=False) if len(table) else table

left_table = show_leftovers(results, mn_general, g_mn_left, g_oc_left, keys, "Year")
print("MyNeta general rows without a partner:", len(left_table), "| of them winners:", int(left_table["MyNeta winner"].sum()))
print(left_table.head(25).to_string(index=False))

MyNeta general rows without a partner: 262 | of them winners: 0
 Year                 Seat                                  MyNeta name                 MyNeta party  MyNeta winner                     Best OpenCity name    OC party  similarity
 2014             VARANASI                             RAM LAKHAN GUPTA              Jantantra Party          False                       RAM LAKHAN GUPTA         JAP        1.00
 2019      CHENNAI CENTRAL                          JITENDRA KUMAR JAIN                          RPI          False                    JITENDRA KUMAR JAIN      RPI(A)        1.00
 2019            MADHUBANI                                SHAKEEL AHMAD                          INC          False                          SHAKEEL AHMAD         IND        1.00
 2019               BALLIA                MAJOR RAMESH CHANDRA UPADHYAY Akhil Bharat Hindu Mahasabha          False          MAJOR RAMESH CHANDRA UPADHYAY         IND        1.00
 2019         DOMARIYAGANJ               

### Step 8: build the merged general election table

In [17]:
mn_cols = {"candidate": "MN_Candidate", "party": "MN_Party", "education": "MN_Education", "age": "MN_Age",
           "criminal_cases": "MN_Criminal_Cases", "Assets_Rs": "Assets_Rs", "Liabilities_Rs": "Liabilities_Rs",
           "Assets_Rs_Status": "Assets_Status", "Liabilities_Rs_Status": "Liabilities_Status",
           "Assets_Nil": "Assets_Nil", "is_winner": "MN_Is_Winner", "MN_Row": "MN_Row"}

def attach(oc_frame, mn_frame, pairs):
    link = pd.DataFrame(pairs, columns=["oc", "mn", "Match_Rule", "Match_Score"]).set_index("oc")
    out = oc_frame.join(link[["Match_Rule", "Match_Score"]])
    mn_part = mn_frame.loc[link["mn"], list(mn_cols)].rename(columns=mn_cols)
    mn_part.index = link.index
    out = out.join(mn_part)
    out["Has_MyNeta"] = out["MN_Row"].notna()
    out["Has_Assets"] = out["Assets_Rs"].notna()
    assert len(out) == len(oc_frame)
    return out

general = attach(results, mn_general, g_pairs).rename(columns={"Year": "Election_Year"})
general["Term"] = general["Election_Year"]
general["Is_ByElection"] = False
general["OC_File"] = "opencity_results_clean"
general["Bye_Year"] = np.nan
print("general rows:", len(general), "| with MyNeta:", int(general["Has_MyNeta"].sum()))
print(general.groupby(["Election_Year", "State"])["Has_MyNeta"].mean().unstack().round(3).to_string())

general rows: 10378 | with MyNeta: 9966
State          Bihar  Maharashtra  Tamil Nadu  Uttar Pradesh
Election_Year                                               
2014           0.967        0.931       0.989          0.946
2019           0.952        0.912       0.955          0.939
2024           0.988        0.973       0.994          0.994


### Step 9: by-elections

The OpenCity by-election file ends in 2021, so MyNeta's 2022 (Azamgarh, Rampur, Mainpuri) and 2024 (Nanded) by-elections have no partner and are reported. NOTA is dropped; position and margin are recomputed.

In [18]:
bye = oc_bye[oc_bye["State"].isin(STATES) & (oc_bye["Poll_No"] > 0)].copy()
bye["Is_NOTA"] = bye["Party"].eq("NOTA") | bye["Candidate"].str.upper().isin(["NOTA", "NONE OF THE ABOVE"])
print("OpenCity by-election rows in the four states:", len(bye), "| NOTA rows removed:", int(bye["Is_NOTA"].sum()))
bye = bye[~bye["Is_NOTA"]].copy()
assert bye["Votes"].notna().all()
bye = add_name_keys(bye, "Candidate")
bye = bye.rename(columns={"Year": "Bye_Year", "Valid_Votes": "Seat_Valid_Votes"})
seat_names = dict(zip(zip(seats["State"], seats["Constituency_No"]), seats["Constituency_Name"]))
bye["Constituency_Name"] = [seat_names.get(k, n) for k, n in zip(zip(bye["State"], bye["Constituency_No"]), bye["Constituency_Name"])]

seat_cols = ["State", "Constituency_No", "Bye_Year"]
bye["Position"] = bye.groupby(seat_cols)["Votes"].rank(method="min", ascending=False).astype(int)
bye["Winner"] = bye["Position"] == 1
first = bye[bye["Position"] == 1].set_index(seat_cols)["Votes"]
second = bye[bye["Position"] == 2].set_index(seat_cols)["Votes"]
bye = bye.drop(columns="Margin").merge((first - second).rename("Margin"), left_on=seat_cols, right_index=True, how="left")
bye["Votes"] = bye["Votes"].astype(int)
bye["Margin"] = bye["Margin"].astype("Int64")
bye["Incumbent"] = bye["Incumbent"].map(lambda v: np.nan if pd.isna(v) else str(v) == "True")
assert bye.groupby(seat_cols)["Winner"].sum().eq(1).all()
print("by-election seats:", bye.groupby(seat_cols).ngroups, "| candidate rows:", len(bye))

mn_bye = myneta[myneta["is_byelection"]].copy()
mn_bye["Constituency_No"] = mn_bye["Constituency_No"].astype(int)
mn_bye["Bye_Year"] = pd.to_datetime(mn_bye["bye_election_date"], format="%d-%m-%Y").dt.year
print("MyNeta by-election rows:", len(mn_bye), "| seats:", mn_bye.groupby(seat_cols).ngroups)

OpenCity by-election rows in the four states: 143 | NOTA rows removed: 9
by-election seats: 12 | candidate rows: 134
MyNeta by-election rows: 167 | seats: 16


In [19]:
b_pairs, b_mn_left, b_oc_left = run_matching(bye, mn_bye, seat_cols)
print("matched pairs:", len(b_pairs), "| MyNeta rows left:", len(b_mn_left), "| OpenCity rows left:", len(b_oc_left))
oc_events, mn_events = set(bye.groupby(seat_cols).groups), set(mn_bye.groupby(seat_cols).groups)
print("only in MyNeta:", sorted(mn_events - oc_events))
print("only in OpenCity:", sorted(oc_events - mn_events))
bc = pd.DataFrame(b_pairs, columns=["oc", "mn", "rule", "score"])
print("winner flag agrees:", (bye.loc[bc["oc"], "Winner"].values == mn_bye.loc[bc["mn"], "is_winner"].values).mean())
inside = [m for m in b_mn_left if (mn_bye.at[m, "State"], mn_bye.at[m, "Constituency_No"], mn_bye.at[m, "Bye_Year"]) in oc_events]
print("\nleft over inside by-elections found on both sides:")
print(show_leftovers(bye, mn_bye, inside, b_oc_left, seat_cols, "Bye_Year").to_string(index=False) if inside else "none")

byelection = attach(bye, mn_bye, b_pairs)
byelection["Election_Year"] = byelection["Bye_Year"]
byelection["Term"] = np.where(byelection["Bye_Year"] <= 2018, 2014, 2019)
byelection["Is_ByElection"] = True
byelection["OC_File"] = "opencity_byelections_2014_onwards"
byelection["Party_Full"] = byelection["Party_Raw"]
print("\nby-election rows:", len(byelection), "| with MyNeta:", int(byelection["Has_MyNeta"].sum()))

matched pairs: 118 | MyNeta rows left: 49 | OpenCity rows left: 16
only in MyNeta: [('Maharashtra', 16, 2024), ('Uttar Pradesh', 7, 2022), ('Uttar Pradesh', 21, 2022), ('Uttar Pradesh', 69, 2022)]
only in OpenCity: []
winner flag agrees: 1.0

left over inside by-elections found on both sides:
 Year     Seat        MyNeta name                   MyNeta party  MyNeta winner             Best OpenCity name OC party  similarity
 2018  KAIRANA      PRITI KASHYAP      Jay Hind Jay Bharat Party          False           KUMARI PRITI KASHYAP    JHJBP        0.79
 2018  PHULPUR    RAIS AHMAD KHAN         Parivartan Samaj Party          False       PATRAKAR RAIS AHMAD KHAN      PSP        0.77
 2018  PHULPUR    VARUN KUMAR PAL                            PSP          False                  VARUN DEV PAL     PSSP        0.71
 2018  PHULPUR             RAMESH Lohia Sanyukta Samajwadi Party          False                   RAMESH KUMAR     LSSP        0.67
 2014 MAINPURI ASHOK KUMAR TEWARI             

### Step 10: combine and set types

In [20]:
columns = ["OC_File", "OC_Row", "MN_Row", "State", "Constituency_No", "Constituency_Name", "Constituency_Type", "Term", "Election_Year",
           "Is_ByElection", "Bye_Year", "Person_ID", "Candidate", "Candidate_Key", "Namesake_In_Seat", "Sex", "Incumbent",
           "Party", "Party_Full", "Party_ID", "Party_Canonical", "Party_Type", "Party_Method", "Party_Group",
           "Position", "Winner", "Votes", "Vote_Share_Percentage", "Margin", "Seat_Valid_Votes",
           "MN_Candidate", "MN_Party", "MN_Education", "MN_Age", "MN_Criminal_Cases", "Assets_Rs", "Liabilities_Rs",
           "Assets_Status", "Liabilities_Status", "Assets_Nil", "Has_MyNeta", "Has_Assets", "Match_Rule", "Match_Score"]

INT_COLS = ["OC_Row", "MN_Row", "Constituency_No", "Term", "Election_Year", "Bye_Year", "Person_ID", "Party_ID", "Position",
            "Votes", "Margin", "Seat_Valid_Votes", "MN_Age", "MN_Criminal_Cases", "Match_Rule"]
FLOAT_COLS = ["Vote_Share_Percentage", "Assets_Rs", "Liabilities_Rs", "Match_Score"]
BOOL_COLS = ["Is_ByElection", "Namesake_In_Seat", "Incumbent", "Winner", "MN_Is_Winner", "Assets_Nil", "Has_MyNeta", "Has_Assets"]

def set_types(frame):
    frame = frame.reindex(columns=columns).copy()
    for c in INT_COLS:
        frame[c] = pd.to_numeric(frame[c]).round().astype("Int64")
    for c in FLOAT_COLS:
        frame[c] = pd.to_numeric(frame[c]).astype("float64")
    for c in ["Is_ByElection", "Winner", "Has_MyNeta", "Has_Assets"]:
        frame[c] = frame[c].astype(bool)
    for c in ["Namesake_In_Seat", "Incumbent", "Assets_Nil"]:
        frame[c] = frame[c].astype("boolean")
    return frame

general_out = set_types(general)
bye_out = set_types(byelection)
all_out = pd.concat([general_out, bye_out], ignore_index=True)

unmatched = pd.concat([mn_general.loc[g_mn_left].assign(Reason="no partner in seat"),
                       mn_bye.loc[b_mn_left].assign(Reason=lambda d: np.where(
                           [(s, n, y) in oc_events for s, n, y in zip(d["State"], d["Constituency_No"], d["Bye_Year"])],
                           "no partner in seat", "by-election not in OpenCity"))])
unmatched["year"] = unmatched["Year"].fillna(unmatched["year"]).astype(int) if "Year" in unmatched else unmatched["year"]
unmatched = unmatched[["MN_Row", "year", "State", "const", "Constituency_No", "candidate", "party", "Party_Canonical",
                       "is_winner", "is_byelection", "bye_election_date", "Assets_Rs", "Reason"]]
print("general:", general_out.shape, "| by-election:", bye_out.shape, "| combined:", all_out.shape, "| unmatched MyNeta:", unmatched.shape)

general: (10378, 44) | by-election: (134, 44) | combined: (10512, 44) | unmatched MyNeta: (311, 13)


### Step 11: checks on the output

Every check below is an `assert`, so the notebook stops if one fails.

In [21]:
# 1. row counts
assert len(general_out) == N_RESULTS, "general table lost or gained rows"
assert general_out["OC_Row"].is_unique and bye_out["OC_Row"].is_unique
matched_mn = all_out["MN_Row"].dropna()
assert matched_mn.is_unique, "a MyNeta row was used twice"
assert len(matched_mn) + len(unmatched) == N_MYNETA, "MyNeta rows lost"
assert set(matched_mn).isdisjoint(set(unmatched["MN_Row"]))
print(f"1. rows: general {len(general_out)} = OpenCity {N_RESULTS}; MyNeta matched {len(matched_mn)} + unmatched {len(unmatched)} = {N_MYNETA}")

# 2. one winner per seat and election
w = all_out.groupby(["Is_ByElection", "Election_Year", "State", "Constituency_No"])["Winner"].sum()
assert w.eq(1).all()
print("2. exactly one winner in each of", len(w), "seat-elections")

# 3. matched rows agree on the winner flag
m = all_out[all_out["Has_MyNeta"]]
agree = (m["Winner"].values == myneta.loc[m["MN_Row"].astype(int), "is_winner"].values).mean()
assert agree == 1.0
print("3. winner flag agrees on every matched pair:", agree)

# 4. every winner has MyNeta data (except by-elections missing on one side)
gw = general_out[general_out["Winner"]]
print("4. general-election winners with MyNeta data:", int(gw["Has_MyNeta"].sum()), "of", len(gw))
no_mn_winner = mn_general.groupby(keys)["is_winner"].sum()
no_mn_winner = set(no_mn_winner[no_mn_winner == 0].index)
missing_w = gw[~gw["Has_MyNeta"]]
print("   winners without MyNeta data (MyNeta scrape has no winner row for these seats):")
print(missing_w[["Election_Year", "State", "Constituency_Name", "Candidate", "Party"]].to_string(index=False))
assert set(zip(missing_w["Election_Year"], missing_w["State"], missing_w["Constituency_No"])) == no_mn_winner

1. rows: general 10378 = OpenCity 10378; MyNeta matched 10084 + unmatched 311 = 10395
2. exactly one winner in each of 633 seat-elections
3. winner flag agrees on every matched pair: 1.0
4. general-election winners with MyNeta data: 618 of 621
   winners without MyNeta data (MyNeta scrape has no winner row for these seats):
 Election_Year         State Constituency_Name                    Candidate Party
          2014   Maharashtra MUMBAI NORTH WEST GAJANAN CHANDRAKANT KIRTIKAR   SHS
          2014 Uttar Pradesh       FARRUKHABAD                MUKESH RAJPUT   BJP
          2019 Uttar Pradesh          AKBARPUR         DEVENDRA SINGH BHOLE   BJP


In [22]:
# 5. numeric types
expected = {**{c: "Int64" for c in INT_COLS}, **{c: "float64" for c in FLOAT_COLS}}
wrong = {c: str(all_out[c].dtype) for c, t in expected.items() if str(all_out[c].dtype) != t}
assert not wrong, wrong
assert (all_out["Assets_Rs"].dropna() >= 0).all() and (all_out["Liabilities_Rs"].dropna() >= 0).all()
assert (all_out["Votes"] >= 0).all() and all_out["Vote_Share_Percentage"].between(0, 100).all()
print("5. numeric columns have the expected types; money, votes and shares are in range")
print(all_out.dtypes.astype(str).value_counts().to_dict())

# 6. asset values: every parsed value agrees with MyNeta's approximation or has a known reason
st = all_out.loc[all_out["Has_MyNeta"], "Assets_Status"].value_counts().to_dict()
print("6. asset status of matched rows:", st, "| liabilities:", all_out.loc[all_out["Has_MyNeta"], "Liabilities_Status"].value_counts().to_dict())
assert not set(st) & {"mismatch", "unreadable", "missing"}

# 7. no messy text left in the output
messy = {}
for c in text_columns(all_out):
    s = all_out[c].dropna().astype(str)
    n = int((s != s.str.strip()).sum() + s.str.contains(r"\s{2,}|[ \n\t]").sum())
    if n:
        messy[c] = n
assert not messy, messy
print("7. no leading/trailing/double spaces, NBSP or line breaks in any text column")

5. numeric columns have the expected types; money, votes and shares are in range
{'object': 18, 'Int64': 15, 'bool': 4, 'float64': 4, 'boolean': 3}
6. asset status of matched rows: {'ok': 9996, 'nil': 83, 'no_approx': 5} | liabilities: {'no_approx': 5576, 'ok': 4508}
7. no leading/trailing/double spaces, NBSP or line breaks in any text column


In [23]:
# 8. nulls: MyNeta columns are empty exactly when the row has no MyNeta partner
mn_out_cols = ["MN_Row", "MN_Candidate", "MN_Party", "MN_Education", "MN_Criminal_Cases", "Assets_Rs", "Liabilities_Rs", "Match_Rule", "Match_Score"]
for c in mn_out_cols:
    assert (all_out[c].isna() == ~all_out["Has_MyNeta"]).all(), c
nulls = all_out.isna().sum()
nulls = nulls[nulls > 0].to_frame("nulls")
nulls["share"] = (nulls["nulls"] / len(all_out)).round(3)
nulls["why"] = ""
nulls.loc[nulls.index.isin(mn_out_cols + ["MN_Age", "MN_Is_Winner", "Assets_Status", "Liabilities_Status", "Assets_Nil"]), "why"] = "no MyNeta partner"
nulls.loc["MN_Age", "why"] = "no MyNeta partner, or age 0 in MyNeta" if "MN_Age" in nulls.index else ""
nulls.loc[nulls.index.isin(["Party_ID", "Party_Canonical"]), "why"] = "party not in TCPD (unmapped)"
nulls.loc[nulls.index.isin(["Bye_Year"]), "why"] = "general election rows"
nulls.loc[nulls.index.isin(["Sex"]), "why"] = "2024 file has no sex column"
nulls.loc[nulls.index.isin(["Person_ID", "Candidate_Key", "Namesake_In_Seat", "Party_Group"]), "why"] = "by-election rows (column only in the general file)"
nulls.loc["Sex", "why"] = "2024 file has no sex column (+ by-election gaps)"
print("8. nulls in the combined table:")
print(nulls.to_string())

8. nulls in the combined table:
                    nulls  share                                                 why
MN_Row                428  0.041                                   no MyNeta partner
Bye_Year            10378  0.987                               general election rows
Person_ID             134  0.013  by-election rows (column only in the general file)
Candidate_Key         134  0.013  by-election rows (column only in the general file)
Namesake_In_Seat      134  0.013  by-election rows (column only in the general file)
Sex                  3503  0.333    2024 file has no sex column (+ by-election gaps)
Party_ID              511  0.049                        party not in TCPD (unmapped)
Party_Canonical       511  0.049                        party not in TCPD (unmapped)
Party_Group           134  0.013  by-election rows (column only in the general file)
MN_Candidate          428  0.041                                   no MyNeta partner
MN_Party              428  0.041 

In [24]:
# 9. coverage: share of OpenCity rows with MyNeta data, and of MyNeta rows with an OpenCity partner
cov = general_out.groupby(["Election_Year", "State"]).agg(oc_rows=("OC_Row", "size"), with_myneta=("Has_MyNeta", "sum"))
cov["share"] = (cov["with_myneta"] / cov["oc_rows"]).round(3)
print("9. coverage of general election rows:")
print(cov.to_string())
print("\nMyNeta rows matched:", round(len(matched_mn) / N_MYNETA, 3))

# 10. keys that did not match
print("\n10. keys that did not match")
print("  seat names matched only by similar spelling:", fuzzy_seats)
print("  unmatched MyNeta rows by reason:", unmatched["Reason"].value_counts().to_dict())
oc_left = general_out[~general_out["Has_MyNeta"]]
print("  OpenCity general rows without MyNeta:", len(oc_left), "| their total vote share:",
      round(oc_left["Votes"].sum() / general_out["Votes"].sum() * 100, 2), "% of all votes")
print("  highest-placed of them:")
print(oc_left.sort_values("Vote_Share_Percentage", ascending=False)[["Election_Year", "State", "Constituency_Name", "Candidate", "Party", "Position", "Vote_Share_Percentage"]].head(10).to_string(index=False))

9. coverage of general election rows:
                             oc_rows  with_myneta  share
Election_Year State                                     
2014          Bihar              607          587  0.967
              Maharashtra        897          835  0.931
              Tamil Nadu         845          836  0.989
              Uttar Pradesh     1288         1218  0.946
2019          Bihar              626          596  0.952
              Maharashtra        867          791  0.912
              Tamil Nadu         850          812  0.955
              Uttar Pradesh      979          919  0.939
2024          Bihar              497          491  0.988
              Maharashtra       1121         1091  0.973
              Tamil Nadu         950          944  0.994
              Uttar Pradesh      851          846  0.994

MyNeta rows matched: 0.97

10. keys that did not match
  seat names matched only by similar spelling: [('Maharashtra', 'AHMEDNAGAR', 'AHMADNAGAR'), ('Maharashtra',

### Step 12: save

In [25]:
general_out.to_csv(OUT_DIR / f"merged_general_{COLLECTION_DATE}.csv", index=False)
bye_out.to_csv(OUT_DIR / f"merged_byelection_{COLLECTION_DATE}.csv", index=False)
all_out.to_csv(OUT_DIR / f"merged_all_{COLLECTION_DATE}.csv", index=False)
unmatched.to_csv(OUT_DIR / f"merged_unmatched_myneta_{COLLECTION_DATE}.csv", index=False)
party_map.drop(columns="Options").to_csv(OUT_DIR / f"party_map_{COLLECTION_DATE}.csv", index=False)

# read back and confirm the saved file has the same shape and values
back = pd.read_csv(OUT_DIR / f"merged_all_{COLLECTION_DATE}.csv")
assert back.shape == all_out.shape
assert np.allclose(back["Assets_Rs"].fillna(-1), all_out["Assets_Rs"].fillna(-1))
print("saved to", OUT_DIR, "| rows:", len(general_out), "general,", len(bye_out), "by-election,", len(unmatched), "unmatched MyNeta")

saved to /Users/pravas/Desktop/DA25B051/DCV-Project/clean | rows: 10378 general, 134 by-election, 311 unmatched MyNeta


In [26]:
# quick look: median assets of winners and others (rupees crore)
view = general_out[general_out["Has_Assets"]].assign(Assets_Crore=lambda d: d["Assets_Rs"] / 1e7)
print(view.groupby(["Election_Year", "Winner"])["Assets_Crore"].agg(["size", "median"]).round(2).unstack().to_string())

               size       median      
Winner        False True   False True 
Election_Year                         
2014           3271   205   0.21  3.39
2019           2912   206   0.23  6.58
2024           3165   207   0.27  9.37
